# Batch Data Processing from an API

In this lab, you will learn how to interact with the Open Library API and extract data in a batch way. You will explore what pagination means and how to send API requests without requiring authentication.

# Table of Contents


- [ About the Open Library API](#about)
- [ 1 - Understand the Basics of APIs](#1)
  - [ 1.1 - Get Books by Subject](#1-1)
    - [ Exercise 1](#ex01)
  - [ 1.2 - Pagination](#1-2)
    - [ Exercise 2](#ex02)
- [ 2 - Batch Pipeline](#2)
  - [ Exercise 3](#ex03)
  - [ Exercise 4](#ex04)
- [ 3 - Optional - API Rate Limits](#3)

<a id='about'></a>
## About the Open Library API

[Open Library](https://openlibrary.org/) is a free, open catalog of books. The API is public and requires **no API key or authentication** to read.

### Key terms

| Term | What it is | Example |
|------|------------|---------|
| **Work** | A book in the catalog | "Hamlet" by Shakespeare |
| **Author** | A person who wrote one or more works | William Shakespeare |
| **Subject** | A topic that groups related works | `databases`, `programming` |
| **`work_key`** | The internal identifier for a work, of the form `/works/<work_id>` | `/works/OL777826W` |
| **`author_key`** | The internal identifier for an author, of the form `/authors/<author_id>` | `/authors/OL64552A` |
| **`subject_key`** | The internal identifier for a subject, of the form `/subjects/<subject>` | `/subjects/databases` |

> **Note on Work.** In the Open Library API, a *Work* is the book itself. The different printings, translations, and formats of that book are all linked to the same Work.

### Endpoints used in this lab

| Endpoint | Returns |
|----------|---------|
| `GET /subjects/<subject>.json` | The works tagged with the given subject |
| `GET /authors/<author_id>.json` | Details about a single author |
| `GET /authors/<author_id>/works.json` | The works written by a given author |

### Base URL and full endpoint URLs

The base URL of the API is `https://openlibrary.org`. You build the full URL of an endpoint by appending the path above to it:

| What you want | Full URL | Docs (parameters & response format) |
|---------------|----------|-------------------------------------|
| Works for a subject | `https://openlibrary.org/subjects/<subject>.json` | [Subjects API](https://openlibrary.org/dev/docs/api/subjects) |
| Details for an author | `https://openlibrary.org/authors/<author_id>.json` | [Authors API](https://openlibrary.org/dev/docs/api/authors) |
| Works written by an author | `https://openlibrary.org/authors/<author_id>/works.json` | [Authors API](https://openlibrary.org/dev/docs/api/authors) |

For example, the full URL to get works for the `databases` subject is `https://openlibrary.org/subjects/databases.json`, and the full URL for the author with id `OL64552A` is `https://openlibrary.org/authors/OL64552A.json`.

Each endpoint can also accept additional query parameters, which are **appended to the full URL** as a query string (starting with `?`, with `&` between multiple parameters). For example, the Subjects endpoint supports `published_in` to filter by year range:

```
https://openlibrary.org/subjects/databases.json?published_in=2020-2026
```

The full list of parameters each endpoint accepts, along with the exact response format, is described in the docs linked above.

Since each API has its own conventions, it's a good habit to read the docs for any endpoint you work with. This lab includes several documentation links — feel free to skim them now or revisit later.

Each endpoint returns JSON. In Python, calling `.json()` on the `requests` response parses the body into a `dict`.

<a id='1'></a>
## 1 - Understand the Basics of APIs

Several packages in Python allow you to request data from an API; in this lab, you will use the `requests` package, which is a popular and versatile library to perform HTTP requests. It provides a simple and easy-to-use way to interact with web services and APIs. Let's load the required packages:

In [1]:
from typing import Dict, Any, Callable

import json
import requests

<a id='1-1'></a>
### 1.1 - Get Books by Subject

Let's fetch all works tagged with the `databases` subject that were published between 2020 and 2026.

For that, you are provided with the `get_works_by_subject` function defined in the next code cell. The function takes a `url` plus two parameters (referred to as pagination parameters):
- `offset`: the index at which to start (0 = first item).
- `limit`: the maximum number of items to return in this request.

The function appends the pagination parameters to the full URL as query parameters (e.g. `?offset=0&limit=20`).

<a id='ex01'></a>
### Exercise 1

Follow these instructions to complete the function `get_works_by_subject` in the next cell:
1. The `request_url` variable holds the full URL with `offset` and `limit` appended as query parameters. Use `request_url` to perform a `get()` request with the `requests` library.
2. The result is a `Response` object. Call its `.json()` method to parse the response body into a Python dictionary, and return that dictionary.

In [2]:
def get_works_by_subject(url: str, offset: int=0, limit: int=20) -> Dict[Any, Any]:
    """Perform get() request to subject endpoint

    Args:
        url (str): Base url for the request
        offset (int, optional): Page offset for pagination. Defaults to 0.
        limit (int, optional): Number of elements per page. Defaults to 20.

    Returns:
        Dict[Any, Any]: Request response parsed as a Python dictionary
    """
    if "?" in url:
        request_url = f"{url}&offset={offset}&limit={limit}"
    else:
        request_url = f"{url}?offset={offset}&limit={limit}"
        
    ### START CODE HERE ### (~ 2 lines of code)
    try:
        # Perform a get() request using the request_url 
        response = requests.get(url=request_url)
        # Call json() on the response to return it as a Python dictionary
        return response.json()
    ### END CODE HERE ###

    except Exception as err:
        print(f"Error requesting data: {err}")
        return {'error': err}

Now call the function with `URL_SUBJECT_DATABASES`, which points to the works under the `databases` subject.

In [3]:
URL_SUBJECT_DATABASES = 'https://openlibrary.org/subjects/databases.json?published_in=2020-2026'
subjects_response = get_works_by_subject(URL_SUBJECT_DATABASES)

The response has been parsed into a Python dictionary. Let's explore its structure:

In [4]:
subjects_response.keys()

dict_keys(['key', 'name', 'subject_type', 'solr_query', 'work_count', 'works'])

The Subjects response includes a `'work_count'` field showing the **total** number of works under that subject.

In [5]:
subjects_response.get('work_count')

94

If you ask for items past `work_count` (for example with a large `offset`), the response still comes back with the metadata, just with an empty `works` list:

In [6]:
get_works_by_subject(URL_SUBJECT_DATABASES, offset=120, limit=10)

{'key': '/subjects/databases',
 'name': 'databases',
 'subject_type': 'subject',
 'solr_query': 'subject_key:"databases"',
 'work_count': 94,
 'works': []}

In [7]:
subjects_response['work_count']

94

You can explore the returned works using the `'works'` field. This will return a list of items, each containing a dictionary with information for each work. You can take a look at the number of items returned:

In [8]:
len(subjects_response.get('works'))

20

Explore the contents of the third work:

In [9]:
for key, val in subjects_response.get('works')[2].items():
    print(f"{key}: {val}")

key: /works/OL25863177W
title: Machine Learning and Knowledge Discovery in Databases
edition_count: 11
cover_id: None
cover_edition_key: None
subject: ['Machine learning', 'Data mining', 'Databases']
ia_collection: []
printdisabled: False
lending_edition: 
lending_identifier: 
authors: [{'key': '/authors/OL7117457A', 'name': 'Albert Bifet'}, {'key': '/authors/OL7201406A', 'name': 'Michael May'}, {'key': '/authors/OL7873069A', 'name': 'Bianca Zadrozny'}, {'key': '/authors/OL7873070A', 'name': 'Ricard Gavalda'}, {'key': '/authors/OL7833855A', 'name': 'Dino Pedreschi'}]
first_publish_year: 2015
ia: None
public_scan: False
has_fulltext: False


Notice the `authors` field above: each work has a list of authors, and each one has a `key` of the form `/authors/<author_id>`. These keys are the identifiers we'll use later in Section 2 to look up each author's details. Let's extract them from this first work:

In [10]:
first_work = subjects_response.get('works')[2]
[author.get('key') for author in first_work.get('authors')]

['/authors/OL7117457A',
 '/authors/OL7201406A',
 '/authors/OL7873069A',
 '/authors/OL7873070A',
 '/authors/OL7833855A']

So far you've made a single request that returned just one page of works. To collect **all** of them, you need pagination, covered next.

<a id='1-2'></a>
### 1.2 - Pagination

The `databases` subject has 85 works in total, but only 20 came back in your last request (20 is the `limit` you set). To collect **all** the works, you call the endpoint repeatedly, advancing `offset` by `limit` each time until every page has been fetched. This pattern is called **pagination**.

For that, you are provided with the `paginated_works_by_subject` function in the next code cell. Two of its parameters are worth highlighting:
- `endpoint_request`: the function used to fetch a single page (e.g. `get_works_by_subject` from Exercise 1). Passing the endpoint-calling function as a parameter lets the same pagination logic be reused for any endpoint.
- `offset`: sets the **starting** index for the first request. Inside the function, you will update it in a loop to step through the remaining pages.

The function returns a list with all the collected items.

<a id='ex02'></a>
### Exercise 2

Follow these instructions to complete the `paginated_works_by_subject` function in the next cell, inside the `### START CODE HERE ###` block:

1. Build a `kwargs` dictionary with three keys: `'url'`, `'offset'`, `'limit'`, using the function's parameters.
   
2. Make the **first** request: call `endpoint_request(**kwargs)` and assign the result to `response`.

3. Extend `responses` with the `works` list from `response`. (See the [Subjects API](https://openlibrary.org/dev/docs/api/subjects) docs for the response structure.)

4. Read the total number of works from `response.get('work_count')` and store it in `total_elements`.

5. Loop with `while offset < total_elements - limit:`. Subtracting `limit` stops the loop one page early, so you don't make an empty request once there's nothing left to fetch.

6. Inside the loop:
   - Advance `offset` by `limit`.
   - Update `kwargs["offset"]` to the new value.
   - Call `endpoint_request(**kwargs)` again and extend `responses` with the new `works`.

In [11]:
def paginated_works_by_subject(endpoint_request: Callable, url: str, offset: int=0, limit: int=20) -> list:
    """Performs pagination over an API request done by the endpoint_request function

    Args:
        endpoint_request (Callable): Function that performs the API Calls
        url (str): Endpoint's URL for the request
        offset (int, optional): Offset of the page's request. Defaults to 0.
        limit (int, optional): Limit of the page's request. Defaults to 20.

    Returns:
        list: List with the requested items
    """

    responses = []

    ### START CODE HERE ### (~ 13 lines of code)
    # Create a dictionary named kwargs with the values corresponding to the keys url, offset, limit
    kwargs = { 
            "url": url,
            "offset": offset,
            "limit": limit,
        } 

    
    # Call the endpoint_request() function with the arguments specified in the kwargs dictionary.
    response = endpoint_request(**kwargs)
    # Use extend() method to add the works items to the list of responses.
    responses.extend(response.get('works'))
    # Get the total number of elements in works and save it in the variable total_elements.
    total_elements = response.get('work_count')

    # Run the loop as long as the offset value is smaller than total_elements minus the limit.
    while offset < total_elements - limit:
        # Update the offset value with the current value plus the limit value.
        offset = offset + limit
        # Update the offset value in the kwargs dictionary
        kwargs["offset"] = offset
        
        # Call the endpoint_request() function with the arguments specified in the kwargs dictionary.
        response = endpoint_request(**kwargs)
        # Use extend() method to add the works items to the list of responses.
        responses.extend(response.get('works'))
    ### END CODE HERE ###
        
        print(f"Finished iteration for page with offset: {offset}")

    return responses

Now, execute the `paginated_works_by_subject` with the function `get_works_by_subject` as the `endpoint_request` callable parameter. Use the same URL used in the previous `get_works_by_subject` call. Set the initial `offset` as 0. For the limit, the default value is 20 but you can play with other values if you want.

In [12]:
responses = paginated_works_by_subject(endpoint_request=get_works_by_subject,
                                   url=URL_SUBJECT_DATABASES, 
                                   offset=0, limit=20)

Finished iteration for page with offset: 20
Finished iteration for page with offset: 40
Finished iteration for page with offset: 60
Finished iteration for page with offset: 80


##### __Expected Output__ 
```text
Finished iteration for page with offset: 20
Finished iteration for page with offset: 40
Finished iteration for page with offset: 60
Finished iteration for page with offset: 80
```

Have a look at one of the items:

In [13]:
responses[2]

{'key': '/works/OL25863177W',
 'title': 'Machine Learning and Knowledge Discovery in Databases',
 'edition_count': 11,
 'cover_id': None,
 'cover_edition_key': None,
 'subject': ['Machine learning', 'Data mining', 'Databases'],
 'ia_collection': [],
 'printdisabled': False,
 'lending_edition': '',
 'lending_identifier': '',
 'authors': [{'key': '/authors/OL7117457A', 'name': 'Albert Bifet'},
  {'key': '/authors/OL7201406A', 'name': 'Michael May'},
  {'key': '/authors/OL7873069A', 'name': 'Bianca Zadrozny'},
  {'key': '/authors/OL7873070A', 'name': 'Ricard Gavalda'},
  {'key': '/authors/OL7833855A', 'name': 'Dino Pedreschi'}],
 'first_publish_year': 2015,
 'ia': None,
 'public_scan': False,
 'has_fulltext': False}

In [14]:
len(responses)

94

With the `paginated_works_by_subject` function that you created, you are now able to get all 85 available items.

<a id='2'></a>
## 2 - Batch Pipeline

Now that you have learned the basics of working with APIs, let's build a small pipeline that, for every work in a given subject, extracts information about its authors. The end result is a local JSON file mapping each unique `author_key` to that author's details and full list of works.

In the `src/` folder, you are given two scripts that implement this pipeline:

- [`endpoint.py`](src/endpoint.py) defines the API call functions:

  - `get_paginated_works_by_subject`: already completed for you, similar to `paginated_works_by_subject` from Exercise 2. It uses the Subjects API to fetch works for a given subject.

  - `get_authors`: which you will complete in Exercise 3. For a given `author_key`, it calls the two [Authors API](https://openlibrary.org/dev/docs/api/authors) endpoints:
    - `GET /authors/<author_id>.json`: the author's details.
    - `GET /authors/<author_id>/works.json`: the works written by the author (paginated).
  Recall from the overview that author keys are of the form `/authors/<author_id>`, so the function takes the base URL (`https://openlibrary.org`) and concatenates it with `author_key` (which already includes `/authors/...`), then appends either `.json` or `/works.json` (e.g. `BASE_URL + author_key + ".json"` → `https://openlibrary.org/authors/OL64552A.json`).

- [`main.py`](src/main.py) orchestrates the pipeline: it fetches all works for the `programming` subject (2022–2026), collects each work's `author_keys`, calls `get_authors` for every unique `author_key`, and saves the result to a JSON file.

<a id='ex03'></a>
### Exercise 3

Go to [`src/endpoint.py`](src/endpoint.py). Search for the comment `Exercise 3` and follow the instructions to complete the `get_authors` function.

1. The function blueprint is already provided. Create `author_details_url` and `author_works_url` from `base_url` and `author_key`. Remember: `author_key` already starts with `/authors/`, so concatenate the two directly (**do not add an extra `/`** between them). The `author_works_url` additionally has `/works.json` appended.

2. Perform a GET request to `author_details_url`. Assign the result to `details_response`.

3. Convert `details_response` to JSON using the `.json()` method. Assign the result to `details_response_json`.

4. In the `while` loop, build `works_url` by appending `?offset=<offset>&limit=<limit>` to `author_works_url`.

5. Perform a GET request to `works_url`. Assign the result to `works_response`.

6. Convert `works_response` to JSON using the `.json()` method. Assign the result to `works_response_json`.

7. Extend the `works_data` list with the value from `"entries"` in `works_response_json`.

8. Update `offset` by adding `limit` to its current value.

Save changes in the file `src/endpoint.py`.

<a id='ex04'></a>
### Exercise 4

Go back to [`src/main.py`](src/main.py). Search for the comment `Exercise 4`. Inside the loop that iterates through `author_keys`, there is a call to the `get_authors` function (its result is assigned to the variable `author_data`). Fill in the parameters:

- `base_url`: use the `BASE_URL` constant defined for you at line 10.
- `author_key`: the loop variable.
- Pass the `kwargs` dictionary defined at the start of `main()` (it holds the offset and limit).

Save changes in the file `src/main.py`.

Inside the same `for` loop, the response `author_data` is added to the `author_items` dictionary using the corresponding `author_key` as the key. Finally, after iterating through all authors, the `author_items` dictionary is saved to a JSON file in the local environment. Take a look at the filename format, which takes the current date and time into account to avoid collisions with other files.
Run the following commands in the terminal to run the `main.py` script:

```bash
cd src
python main.py
```
*Note*: To open the terminal, click on Terminal -> New Terminal in the menu:

<img src="images/VSCodeCourseraTerminal.png"  width="600"/>

Once the script is finished, you should be able to see a file named `author_items_<DATETIME>.json` in the `src` folder.

<a id='3'></a>
## 3 - Optional - API Rate Limits

*Note*: This is an optional section.

Another important aspect to consider when working with APIs is rate limits. Rate limiting is a mechanism used by APIs to control the number of requests that a client can make within a specified period of time. It helps prevent abuse or overload of the API by limiting the frequency or volume of requests from a single client. Here's how rate limiting typically works:

- Request Quotas: APIs may enforce a maximum number of requests that a client can make within a given time window, for example, 100 requests per minute.

- Time Windows: The time window specifies the duration over which the request quota is measured. For example, a rate limit of 100 requests per minute means that the client can make up to 100 requests in any 60-second period.

- Response to Exceeding Limits: When a client exceeds the rate limit, the API can respond with an error code (such as 429 Too Many Requests) or a message indicating that the rate limit has been exceeded. This allows clients to adjust their behavior accordingly. Other APIs implement what is known as soft rate limiting, where exceeding the documented limits does not immediately trigger explicit errors, but instead results in degraded performance, throttling, or eventual blocking. This is how Open Library operates.

- Rate Limit Headers: Some APIs may include headers in the response to indicate the client's current rate limit status, such as the number of requests remaining until the limit resets or the time at which the limit will reset.

Rate limiting helps maintain the stability and reliability of APIs by ensuring fair access to resources and protecting against abusive or malicious behavior. It also allows API providers to allocate resources more effectively and manage traffic loads more efficiently.

You can also see more details about the rate limits of the Open Library API in the [documentation](https://openlibrary.org/developers/api#rate-limits). In particular, this API sets a limit of 1 request per second using soft rate limiting. This means you won't necessarily see an explicit error status code, but you might be blocked, resulting in an error or an additional delay between requests.

Below, you are provided with code that benchmarks the API calls; you can play with the number of requests and the request interval to see the average time of a request. If you perform too many requests and violate the rate limits, you might get an error.

*Note*: This code may take a few minutes to run.

In [15]:
import time

# Define the Databases endpoint
endpoint = 'https://openlibrary.org/subjects/databases.json'

# Define the number of requests to make
num_requests = 100

# Define the interval between requests (in seconds)
request_interval = 0.01  # Adjust as needed based on the API rate limit

# Store the timestamps of requests
timestamps = []

# Make repeated requests to the endpoint
for i in range(num_requests):
    # Make a paginated request to avoid caching
    url = f"{endpoint}?offset={i*20}&limit=20"
    try:
        response = requests.get(url=url)

        timestamps.append(time.time())

        # Check the time between requests 
        if i > 0:
            print(f'Time since last request: {timestamps[-1] - timestamps[-2]:.2f} seconds.')

    except requests.RequestException as e:
        print(f'Request {i+1}: Failed with error {e}')
    
    # Wait for the specified interval before making the next request
    time.sleep(request_interval)

# Calculate the time between successful requests
time_gaps = [timestamps[i] - timestamps[i-1] for i in range(1, len(timestamps))]
print(f'Average time between successful requests: {sum(time_gaps) / len(time_gaps):.2f} seconds')

Time since last request: 0.53 seconds.
Time since last request: 0.73 seconds.
Time since last request: 0.67 seconds.
Time since last request: 0.70 seconds.
Time since last request: 0.87 seconds.
Time since last request: 0.75 seconds.
Time since last request: 0.69 seconds.
Time since last request: 0.71 seconds.
Time since last request: 0.90 seconds.
Time since last request: 0.64 seconds.
Time since last request: 0.58 seconds.
Time since last request: 0.58 seconds.
Time since last request: 0.65 seconds.
Time since last request: 0.53 seconds.
Time since last request: 0.67 seconds.
Time since last request: 0.38 seconds.
Time since last request: 0.57 seconds.
Time since last request: 0.53 seconds.
Time since last request: 0.59 seconds.
Time since last request: 0.51 seconds.
Time since last request: 0.54 seconds.
Time since last request: 0.77 seconds.
Time since last request: 0.67 seconds.
Time since last request: 0.59 seconds.
Time since last request: 0.70 seconds.
Time since last request: 

In this lab, you learned the basics of ingesting data from an API. You worked with pagination and batch extraction in a manual way using API endpoints.